# 实践项目 02：胸部 X 射线残差卷积 VAE 图像生成

胸部 X 射线是一张单通道二维图像。残差卷积 VAE 用卷积层提取空间结构，用残差块保留局部细节，再把图像压缩到连续潜空间；解码器从潜变量重建图像或生成新的图像。

本实践从 `NORMAL` 目录中选取胸片。目录名只用于选择图像，不作为分类标签。每张图像统一为单通道 `64×64`，并归一化到 `[-1,1]`。当前固定图像级划分为 1073 张训练图像和 268 张留出图像；两组文件不重叠。你需要补全模型、损失和训练更新，观察输入、重建、潜空间采样、最近邻与插值结果。

Kaggle 是本项目的首选实践入口。打开公开 Notebook 后点击“复制并编辑”保存到自己的账户，再按单元格逐步运行和修改；下载 Notebook 到电脑运行是补充方式。标有 TODO、`None` 占位和“你的回答”的位置需要完成。先看 shape、变量名、注释和检查代码，再填写内容。Notebook 中的数值应来自你实际运行的输出，不要把未运行的线上结果当成自己的结果。


## 数据到结果

输入图像先经过灰度化、缩放和归一化，得到 `[B,1,64,64]` 的张量。固定划分把 1073 张图像用于训练，把 268 张图像留作重建检查；留出文件不会进入训练更新。残差块在不改变 shape 的情况下细化局部结构；四次下采样把空间尺寸变为 `4×4`，线性层把特征变成 `mu` 与 `logvar`。重参数化得到潜变量 `z`，解码器再把它还原成图像。

重建 L1 关注像素差异，梯度 L1 关注相邻像素的变化，KL 项约束潜变量分布。训练完成后，用输入做重建，用潜变量插值和采样观察新的输出。


In [ ]:
from pathlib import Path
import json, random, re
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import utils

# 固定随机种子和训练规模，便于重复得到相近的训练过程和结果。
SEED=20260803
TRAIN_COUNT=1073
HOLDOUT_COUNT=268
BATCH_SIZE=64
EPOCHS=50
LATENT_DIM=64
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
# 某些 Kaggle 镜像会提供较旧的 P100 等 GPU，但当前 PyTorch 未必包含对应的 CUDA kernel。
# 先检查计算能力；不兼容时改用 CPU，避免在第一次前向计算时出现 CUDA kernel 错误。
cuda_usable=False
if torch.cuda.is_available():
    try:
        device_capability=torch.cuda.get_device_capability(0)
        cuda_usable=device_capability[0]>=7
    except Exception:
        cuda_usable=False
if torch.cuda.is_available() and not cuda_usable:
    print('当前 GPU 架构不在本 PyTorch 支持范围内，改用 CPU 运行。')
DEVICE=torch.device('cuda' if cuda_usable else 'cpu')
OUT=Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()/'outputs'
OUT.mkdir(parents=True,exist_ok=True)
print('device:',DEVICE)

## 1. 数据路径与预处理

推荐挂载 `chest-xray-pneumonia`。数据应包含 `train/NORMAL` 和 `train/PNEUMONIA` 目录。本实践只读取 `NORMAL`，不把目录名作为标签。下载到电脑运行时保留同样的目录结构。

使用 EXIF 方向校正后，把每张图像按中心裁切方式缩放为 `64×64`。像素先变为 `[0,1]`，再映射到 `[-1,1]`，因此解码器最后使用 `Tanh()`。固定划分按图像文件进行，不把同一文件同时放进训练集和留出集。


In [ ]:
INPUT_ROOT=Path('/kaggle/input') if Path('/kaggle/input').exists() else Path.cwd()/'data'
def is_valid_image(path):
    return path.is_file() and '__MACOSX' not in path.parts and not path.name.startswith('._') and path.suffix.lower() in {'.jpeg','.jpg'}

candidates=[p for p in INPUT_ROOT.rglob('train') if p.is_dir() and '__MACOSX' not in p.parts and (p/'NORMAL').is_dir()]
assert candidates,'未找到包含 NORMAL 的 chest_xray/train 目录。'
DATA_ROOT=sorted(candidates,key=lambda p:str(p))[0]
all_files=sorted([p for p in (DATA_ROOT/'NORMAL').iterdir() if is_valid_image(p)])
assert len(all_files)>=TRAIN_COUNT+HOLDOUT_COUNT, f'NORMAL 胸片数量不足：{len(all_files)}'

def load_xray(path):
    # 每张图像都走同一套灰度化、裁切和归一化，保证 batch shape 一致。
    with Image.open(path) as image:
        image=ImageOps.exif_transpose(image).convert('L')
        image=ImageOps.fit(image,(64,64),method=Image.Resampling.BILINEAR,centering=(.5,.5))
        array=np.asarray(image,dtype=np.float32)/255.0
    return torch.from_numpy(array).unsqueeze(0)*2.0-1.0

class XrayDataset(Dataset):
    def __init__(self,files): self.files=list(files)
    def __len__(self): return len(self.files)
    def __getitem__(self,index): return load_xray(self.files[index])

# 先固定文件顺序，再按同一个随机排列切出训练集和留出集。
rng=np.random.default_rng(SEED)
order=rng.permutation(len(all_files))
train_files=[all_files[int(i)] for i in order[:TRAIN_COUNT]]
holdout_files=[all_files[int(i)] for i in order[TRAIN_COUNT:TRAIN_COUNT+HOLDOUT_COUNT]]
assert set(train_files).isdisjoint(holdout_files)
train_dataset=XrayDataset(train_files)
holdout_dataset=XrayDataset(holdout_files)
train_loader=DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True,generator=torch.Generator().manual_seed(SEED),num_workers=0,drop_last=True)
holdout_loader=DataLoader(holdout_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0)
# loader 是题目后续训练单元使用的简短别名；它只指向训练集。
loader=train_loader
reference_images=next(iter(train_loader))
holdout_reference_images=next(iter(holdout_loader))
utils.save_image((reference_images[:36]+1)/2,OUT/'task2_real_xray_grid.png',nrow=6)

real_display=((reference_images+1)/2).clamp(0,1)
plt.figure(figsize=(7.2,4.2))
plt.hist(real_display.flatten().numpy(),bins=60,color='#2b7b9b',alpha=.88)
plt.xlabel('pixel intensity [0,1]'); plt.ylabel('count')
plt.title('Real chest X-ray intensity distribution')
plt.tight_layout(); plt.savefig(OUT/'task2_intensity_histogram.png',dpi=160); plt.show()
print('data root:',DATA_ROOT)
print('available NORMAL images:',len(all_files))
print('train:',len(train_dataset),'holdout:',len(holdout_dataset),'shape:',tuple(reference_images.shape),'range:',float(reference_images.min()),float(reference_images.max()))

## 任务 1：解释归一化范围

**需要完成：** 说明真实胸片为何归一化到 **[-1, 1]**，以及 Decoder 末层为什么使用 `Tanh`。

**依据：** 预处理使用 **Normalize([.5],[.5])**，输入显示前通过 **(image+1)/2** 映射回 `[0,1]`；Decoder 输出范围应与训练输入范围匹配。

**检查：** 你的说明应同时覆盖输入范围、Decoder 输出范围和显示范围之间的对应关系。

<!-- ===== 请在此处完成：项目02·任务1 归一化范围解释（开始） ===== -->
**你的回答：**
<!-- ===== 请在此处完成：项目02·任务1 归一化范围解释（结束） ===== -->

## 任务 2：补全残差卷积 VAE 的编码、潜变量和解码

**需要完成：** 补全残差块、下采样块、上采样块以及 `ResidualConvVAE`。输入 `[B,1,64,64]` 经过四次下采样后变为 `[B,256,4,4]`；`to_mu` 和 `to_logvar` 输出 `[B,64]`；重参数化后 `z` 仍为 `[B,64]`；解码器输出 `[B,1,64,64]`。

**依据：** 残差块的两次卷积保持空间尺寸，`Conv2d(..., stride=2)` 将高宽减半，`ConvTranspose2d(..., stride=2)` 将高宽扩大一倍。`std=exp(0.5*logvar)`，再用标准正态噪声得到 `z`。

**检查：** 运行探针后，`mu`、`logvar`、`z` 的 shape 应为 `(2,64)`，重建图像 shape 应为 `(2,1,64,64)`。


In [ ]:
LATENT_DIM=64

# ===== 请在此处完成：项目02·任务2 残差卷积 VAE 模型（开始） =====
# TODO 2：依据上方的 shape 说明，补全残差块、下采样、上采样、mu/logvar、重参数化和 forward。
# 每次下采样都把高和宽减半；四次下采样后 [B,1,64,64] 应变为 [B,256,4,4]。
def group_count(channels):
    return 8 if channels % 8 == 0 else 1

class ResidualBlock(nn.Module):
    """两个 3×3 卷积和一条跳连，输入输出 shape 相同。"""
    def __init__(self,channels):
        super().__init__()
        # TODO 2A：补全归一化、卷积和激活层；通道数不能改变
        self.norm1=None
        self.conv1=None
        self.norm2=None
        self.conv2=None
        self.activation=nn.SiLU(inplace=True)
    def forward(self,x):
        # TODO 2A：先归一化、激活、卷积两次，再加回 residual
        raise NotImplementedError('请补全 ResidualBlock.forward')

class DownBlock(nn.Module):
    def __init__(self,in_channels,out_channels):
        super().__init__()
        # TODO 2B：stride=2 的卷积负责改变空间尺寸，ResidualBlock 负责细化特征
        self.down=None
        self.norm=None
        self.activation=nn.SiLU(inplace=True)
        self.residual=None
    def forward(self,x):
        raise NotImplementedError('请补全 DownBlock.forward')

class UpBlock(nn.Module):
    def __init__(self,in_channels,out_channels):
        super().__init__()
        # TODO 2C：用 ConvTranspose2d 将空间尺寸扩大 2 倍
        self.up=None
        self.norm=None
        self.activation=nn.SiLU(inplace=True)
        self.residual=None
    def forward(self,x):
        raise NotImplementedError('请补全 UpBlock.forward')

class ResidualConvVAE(nn.Module):
    def __init__(self,latent_dim=LATENT_DIM):
        super().__init__()
        # TODO 2D：补全 stem、四个 DownBlock、两个潜变量线性层、from_z 和四个 UpBlock
        self.stem=None
        self.encoder=None
        self.to_mu=None
        self.to_logvar=None
        self.from_z=None
        self.decoder=None
    def encode(self,x):
        # TODO 2E：得到 [B,256,4,4]，展平后分别得到 mu 与 logvar
        raise NotImplementedError('请补全 ResidualConvVAE.encode')
    @staticmethod
    def reparameterize(mu,logvar):
        # TODO 2F：使用 z = mu + exp(0.5*logvar)*epsilon
        raise NotImplementedError('请补全 ResidualConvVAE.reparameterize')
    def decode(self,z):
        # TODO 2G：先把 z 变成 [B,256,4,4]，再逐步还原到 [B,1,64,64]
        raise NotImplementedError('请补全 ResidualConvVAE.decode')
    def forward(self,x):
        # TODO 2H：依次完成 encode、reparameterize 和 decode
        raise NotImplementedError('请补全 ResidualConvVAE.forward')

vae=ResidualConvVAE().to(DEVICE)
with torch.no_grad():
    probe=reference_images[:2].to(DEVICE)
    probe_reconstruction,probe_mu,probe_logvar,probe_z=vae(probe)
print('mu:',tuple(probe_mu.shape),'logvar:',tuple(probe_logvar.shape),'z:',tuple(probe_z.shape))
print('reconstruction:',tuple(probe_reconstruction.shape))
assert tuple(probe_mu.shape)==(2,LATENT_DIM)
assert tuple(probe_logvar.shape)==(2,LATENT_DIM)
assert tuple(probe_z.shape)==(2,LATENT_DIM)
assert tuple(probe_reconstruction.shape)==(2,1,64,64)
# ===== 请在此处完成：项目02·任务2 残差卷积 VAE 模型（结束） =====


## 任务 3：组合重建、边缘与 KL 损失并训练 50 轮

**需要完成：** 计算重建 L1、梯度 L1 与 KL 三项，按给定权重相加；补全每个 batch 的清梯度、反向传播、梯度裁剪和优化器更新，完成 **50 轮**训练。

**依据：** `F.l1_loss` 比较像素或相邻像素差；KL 项为 `-0.5 × mean(1 + logvar - mu² - exp(logvar))`；总损失为 `reconstruction + 0.25 × gradient + beta × KL`，其中 `beta` 在前 10 轮逐步增加到 `0.0005`。

**检查：** `history` 的四个列表长度都应为 50，损失应为有限数值，训练曲线和 JSON 应成功生成。合理利用AI工具理解问题，学习知识并尝试给出适当的解决方案。


In [ ]:
EDGE_WEIGHT=.25
KL_WEIGHT=.0005
KL_WARMUP_EPOCHS=10

def gradient_loss(pred,target):
    """用横向和纵向相邻像素差比较轮廓。"""
    pred_dx=pred[:,:,:,1:]-pred[:,:,:,:-1]
    target_dx=target[:,:,:,1:]-target[:,:,:,:-1]
    pred_dy=pred[:,:,1:,:]-pred[:,:,:-1,:]
    target_dy=target[:,:,1:,:]-target[:,:,:-1,:]
    return .5*(F.l1_loss(pred_dx,target_dx)+F.l1_loss(pred_dy,target_dy))

# ===== 请在此处完成：项目02·任务3A 三项 VAE 损失（开始） =====
# TODO 3A：补全重建 L1、梯度 L1 和 KL 三项，并按 beta warm-up 组合总损失。
def vae_loss(reconstruction,target,mu,logvar,beta):
    # TODO：重建项比较像素，梯度项比较相邻像素，KL 项使用 mu/logvar。
    reconstruction_component=None
    edge_component=None
    kl_component=None
    total_loss=None
    return total_loss,reconstruction_component,edge_component,kl_component
# ===== 请在此处完成：项目02·任务3A 三项 VAE 损失（结束） =====

optimizer=torch.optim.AdamW(vae.parameters(),lr=2e-4,weight_decay=1e-4)
history={'total_loss':[],'reconstruction_l1':[],'edge_l1':[],'kl_loss':[]}

# ===== 请在此处完成：项目02·任务3B 50 轮训练更新（开始） =====
# TODO 3B：补全每个 batch 的清梯度、反向传播、梯度裁剪和参数更新；beta 前 10 轮逐步增加。
for epoch in range(1,EPOCHS+1):
    vae.train()
    beta=KL_WEIGHT*min(1.0,epoch/KL_WARMUP_EPOCHS)
    epoch_values={key:[] for key in history}
    for batch_images in train_loader:
        batch_images=batch_images.to(DEVICE)
        # TODO：清空梯度，前向计算，反向传播，梯度裁剪，再更新参数。
        reconstruction,mu,logvar,_=vae(batch_images)
        total_loss,reconstruction_component,edge_component,kl_component=vae_loss(reconstruction,batch_images,mu,logvar,beta)
        epoch_values['total_loss'].append(float(total_loss.detach().cpu()))
        epoch_values['reconstruction_l1'].append(float(reconstruction_component.detach().cpu()))
        epoch_values['edge_l1'].append(float(edge_component.detach().cpu()))
        epoch_values['kl_loss'].append(float(kl_component.detach().cpu()))
    for key in history:
        history[key].append(float(np.mean(epoch_values[key])))
    print(epoch,history['total_loss'][-1],history['reconstruction_l1'][-1],history['edge_l1'][-1],history['kl_loss'][-1])
# ===== 请在此处完成：项目02·任务3B 50 轮训练更新（结束） =====

assert all(len(values)==EPOCHS for values in history.values())
assert all(np.isfinite(values).all() for values in [np.asarray(v) for v in history.values()])
fig,axes=plt.subplots(1,2,figsize=(10,3.5))
axes[0].plot(range(1,EPOCHS+1),history['total_loss'],label='total'); axes[0].plot(range(1,EPOCHS+1),history['reconstruction_l1'],label='reconstruction'); axes[0].plot(range(1,EPOCHS+1),history['edge_l1'],label='gradient')
axes[0].legend(); axes[0].set_xlabel('epoch'); axes[0].set_ylabel('loss')
axes[1].plot(range(1,EPOCHS+1),history['kl_loss'],label='KL'); axes[1].set_xlabel('epoch'); axes[1].set_ylabel('KL loss'); axes[1].legend()
plt.tight_layout(); plt.savefig(OUT/'task2_training_curve.png',dpi=160); plt.show(); plt.close(fig)
training_result={'model':'ResidualConvVAE','epochs':EPOCHS,'latent_dim':LATENT_DIM,'image_shape':[1,64,64],'normalization':'[-1,1]','loss_terms':['reconstruction_l1','gradient_l1','kl'],'edge_weight':EDGE_WEIGHT,'kl_weight_max':KL_WEIGHT,'kl_warmup_epochs':KL_WARMUP_EPOCHS,'train_images':len(train_dataset),'holdout_images':len(holdout_dataset),'history':history,'seed':SEED}
(OUT/'task2_pytorch_result.json').write_text(json.dumps(training_result,indent=2,ensure_ascii=False),encoding='utf-8')
print('saved:',OUT/'task2_training_curve.png',OUT/'task2_pytorch_result.json')


## 任务 4：潜空间采样、最近邻与插值

**需要完成：** 从潜变量生成图像，保存留出图像重建、生成网格、最近邻和插值结果。最近邻距离使用展平后的图像像素，并与训练集图像比较；插值使用两个输入图像的 `mu` 作为端点。

**依据：** `holdout_reference_images` 来自固定留出集，`decode(z)` 接收 `[N,64]` 的潜变量，输出 `[N,1,64,64]`；显示前把 `[-1,1]` 映射回 `[0,1]`。

**检查：** 结果图和 JSON 文件都应存在，输入与生成输出 shape 应保持一致。合理利用AI工具理解问题，学习知识并尝试给出适当的解决方案。


In [ ]:
# ===== 请在此处完成：项目02·任务4 潜空间采样与图像结果（开始） =====
# TODO 4：完成采样、最近邻、插值以及对应的图片和 JSON 保存
vae.eval()
with torch.no_grad():
    input_images=holdout_reference_images[:12].to(DEVICE)
    input_mu,input_logvar=vae.encode(input_images)
    input_latents=input_mu
    reconstructions=vae.decode(input_latents)
    sampled_latents=torch.randn(128,LATENT_DIM,device=DEVICE)
    generated=vae.decode(sampled_latents)

input_reconstruction_l1=float(F.l1_loss(reconstructions,input_images).cpu())
fig,axes=plt.subplots(2,8,figsize=(12,3.5))
for col in range(8):
    axes[0,col].imshow(((input_images[col].cpu().squeeze()+1)/2).clamp(0,1),cmap='gray')
    axes[1,col].imshow(((reconstructions[col].cpu().squeeze()+1)/2).clamp(0,1),cmap='gray')
    axes[0,col].axis('off'); axes[1,col].axis('off')
axes[0,0].set_ylabel('input'); axes[1,0].set_ylabel('reconstruction')
plt.tight_layout(); plt.savefig(OUT/'task2_reconstruction_grid.png',dpi=160); plt.show(); plt.close(fig)
utils.save_image((generated[:36].cpu()+1)/2,OUT/'task2_generated_samples.png',nrow=6)

train_reference_images=torch.stack([train_dataset[i] for i in range(len(train_dataset))])
reference_flat=train_reference_images.to(DEVICE).flatten(1)
generated_flat=generated.flatten(1)
nearest_distance_matrix=torch.cdist(generated_flat,reference_flat,p=1)
nearest_distances,nearest_indices=nearest_distance_matrix.min(dim=1)
nearest_images=train_reference_images[nearest_indices.cpu()]
displayed_count=min(6,len(generated))
fig,axes=plt.subplots(2,6,figsize=(12,4))
for col in range(displayed_count):
    axes[0,col].imshow(((generated[col].cpu().squeeze()+1)/2).clamp(0,1),cmap='gray')
    axes[1,col].imshow(((nearest_images[col].cpu().squeeze()+1)/2).clamp(0,1),cmap='gray')
    axes[0,col].axis('off'); axes[1,col].axis('off')
for col in range(displayed_count,6):
    axes[0,col].axis('off'); axes[1,col].axis('off')
axes[0,0].set_ylabel('generated'); axes[1,0].set_ylabel('nearest input')
plt.tight_layout(); plt.savefig(OUT/'task2_nearest_neighbors.png',dpi=160); plt.show(); plt.close(fig)

with torch.no_grad():
    interpolation_weights=torch.linspace(0,1,steps=8,device=DEVICE)
    interpolation_latents=torch.stack([torch.lerp(input_mu[0],input_mu[1],weight) for weight in interpolation_weights])
    interpolated=vae.decode(interpolation_latents)
utils.save_image((interpolated.cpu()+1)/2,OUT/'task2_latent_interpolation.png',nrow=8)

pixels_per_image=int(np.prod(reference_images.shape[1:]))
nearest_l1=nearest_distances/pixels_per_image
latent_result={'latent_dim':LATENT_DIM,'sample_count':int(len(sampled_latents)),'nearest_neighbor_count':int(len(nearest_distances)),'nearest_neighbor_l1_mean':float(nearest_l1.mean().cpu()),'nearest_neighbor_l1_min':float(nearest_l1.min().cpu()),'interpolation_steps':int(len(interpolation_weights)),'input_reconstruction_l1':input_reconstruction_l1,'outputs':['task2_reconstruction_grid.png','task2_generated_samples.png','task2_nearest_neighbors.png','task2_latent_interpolation.png']}
(OUT/'task2_latent_result.json').write_text(json.dumps(latent_result,indent=2,ensure_ascii=False),encoding='utf-8')
fig,axes=plt.subplots(2,8,figsize=(12,3.5))
for col in range(8):
    axes[0,col].imshow(((input_images[col].cpu().squeeze()+1)/2).clamp(0,1),cmap='gray')
    axes[1,col].imshow(((generated[col].cpu().squeeze()+1)/2).clamp(0,1),cmap='gray')
    axes[0,col].axis('off'); axes[1,col].axis('off')
axes[0,0].set_ylabel('input'); axes[1,0].set_ylabel('generated')
plt.tight_layout(); plt.savefig(OUT/'task2_input_generated_comparison.png',dpi=160); plt.show(); plt.close(fig)
training_result.update({'input_shape':list(input_images.shape[1:]),'reconstruction_shape':list(reconstructions.shape[1:]),'generated_shape':list(generated.shape[1:]),'outputs':['task2_real_xray_grid.png','task2_intensity_histogram.png','task2_training_curve.png','task2_reconstruction_grid.png','task2_input_generated_comparison.png','task2_generated_samples.png','task2_nearest_neighbors.png','task2_latent_interpolation.png','task2_pytorch_result.json','task2_latent_result.json']})
(OUT/'task2_pytorch_result.json').write_text(json.dumps(training_result,indent=2,ensure_ascii=False),encoding='utf-8')
print('generated:',tuple(generated.shape),'nearest mean L1:',float(nearest_l1.mean()))
print('saved:',OUT/'task2_reconstruction_grid.png',OUT/'task2_generated_samples.png',OUT/'task2_nearest_neighbors.png',OUT/'task2_latent_interpolation.png',OUT/'task2_latent_result.json')
# ===== 请在此处完成：项目02·任务4 潜空间采样与图像结果（结束） =====

## 任务 5：观察输入、重建、生成与潜空间结果

从输入/重建对照、生成网格、最近邻图和插值图中记录可辨认结构、模糊区域、伪影、样本差异和连续变化。结合损失曲线与 JSON 数值，说明图像证据和数值证据是否一致。

**需要完成：** 在下方填写观察记录，至少涉及输入与重建差异、生成样本质量、最近邻比较、插值变化和训练损失。合理利用AI工具理解问题，学习知识并尝试给出适当的解决方案。


**你的回答：**

<!-- ===== 请在此处完成：项目02·任务5 结果检查记录（开始） ===== -->
**你的回答：**

- 输入与重建：
- 生成样本：
- 最近邻比较：
- 潜空间插值：
- 损失与综合判断：
<!-- ===== 请在此处完成：项目02·任务5 结果检查记录（结束） ===== -->